In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=8, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import (S, C, B, IDX_TO_CLASS, CONFIDENCE_THRESHOLD,
                         NMS_IOU_THRESHOLD)
from src.utils import convert_xywh_coords, IoU
from operator import itemgetter

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]

                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False)

                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

def filter_group_sort_preds(decoded_preds):
    sorted_preds = []

    # 1. filter and group remaining predictions by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]

        sorted_preds.append(valid_preds)

    # 2. sort each class's predictions by confidence score
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1))

    return sorted_preds


def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []

    for image in sorted_preds:
        final_img_preds = {}

        for class_name, preds in image.items():
            final_img_preds[class_name] = []

            while preds:
                highest_conf = preds.pop(0)
                final_img_preds[class_name].append(highest_conf)

                preds = [pred for pred in preds if
                         IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

        final_preds.append(final_img_preds)

    return final_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

preds = model(X_batch)
preds.shape

torch.Size([8, 1470])

In [6]:
decoded_preds = decode_preds(preds)
decoded_preds

[[('motorbike',
   -0.017032655492490112,
   tensor(-11.1529, grad_fn=<SubBackward0>),
   tensor(5.0265, grad_fn=<SubBackward0>),
   tensor(25.9001, grad_fn=<AddBackward0>),
   tensor(0.2154, grad_fn=<AddBackward0>)),
  ('motorbike',
   0.02323027576035752,
   tensor(22.4858, grad_fn=<SubBackward0>),
   tensor(43.9605, grad_fn=<SubBackward0>),
   tensor(-17.0359, grad_fn=<AddBackward0>),
   tensor(-33.3145, grad_fn=<AddBackward0>)),
  ('pottedplant',
   -0.05392847433540737,
   tensor(46.4218, grad_fn=<SubBackward0>),
   tensor(-35.9323, grad_fn=<SubBackward0>),
   tensor(28.6370, grad_fn=<AddBackward0>),
   tensor(25.8606, grad_fn=<AddBackward0>)),
  ('pottedplant',
   -0.013486254624694993,
   tensor(-2.8830, grad_fn=<SubBackward0>),
   tensor(9.7092, grad_fn=<SubBackward0>),
   tensor(78.4904, grad_fn=<AddBackward0>),
   tensor(-22.8354, grad_fn=<AddBackward0>)),
  ('boat',
   0.06954850057080009,
   tensor(90.8852, grad_fn=<SubBackward0>),
   tensor(-40.1631, grad_fn=<SubBackward0>

In [7]:
sorted_preds = filter_group_sort_preds(decoded_preds)
sorted_preds

[{},
 {},
 {'pottedplant': [('pottedplant',
    0.4092096754303718,
    tensor(163.4863, grad_fn=<SubBackward0>),
    tensor(58.2064, grad_fn=<SubBackward0>),
    tensor(109.1361, grad_fn=<AddBackward0>),
    tensor(2.4072, grad_fn=<AddBackward0>)),
   ('pottedplant',
    0.6630171130015583,
    tensor(58.4880, grad_fn=<SubBackward0>),
    tensor(-59.6517, grad_fn=<SubBackward0>),
    tensor(79.2529, grad_fn=<AddBackward0>),
    tensor(63.0036, grad_fn=<AddBackward0>))],
  'boat': [('boat',
    0.5382174977064302,
    tensor(101.7207, grad_fn=<SubBackward0>),
    tensor(47.0326, grad_fn=<SubBackward0>),
    tensor(200.2770, grad_fn=<AddBackward0>),
    tensor(-38.5687, grad_fn=<AddBackward0>))],
  'bird': [('bird',
    0.5651410619335415,
    tensor(-11.8667, grad_fn=<SubBackward0>),
    tensor(115.7452, grad_fn=<SubBackward0>),
    tensor(87.3825, grad_fn=<AddBackward0>),
    tensor(148.0901, grad_fn=<AddBackward0>))],
  'train': [('train',
    0.6659734384729461,
    tensor(47.7684, 

In [8]:
import torch 
nums = torch.arange(10)
nums = nums.sort()[0]

print(nums)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [9]:
nums = [num for num in nums if num == 2 or num == 4 or num == 6]
nums

[tensor(2), tensor(4), tensor(6)]

In [10]:
final_preds = []

for image in sorted_preds:
    final_img_preds = {}

    for class_name, preds in image.items():
        final_img_preds[class_name] = []

        while preds:
            highest_conf = preds.pop(0)
            final_img_preds[class_name].append(highest_conf)

            preds = [pred for pred in preds if
                     IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

    final_preds.append(final_img_preds)

In [11]:
final_preds, len(final_preds)

([{},
  {},
  {'pottedplant': [('pottedplant',
     0.4092096754303718,
     tensor(163.4863, grad_fn=<SubBackward0>),
     tensor(58.2064, grad_fn=<SubBackward0>),
     tensor(109.1361, grad_fn=<AddBackward0>),
     tensor(2.4072, grad_fn=<AddBackward0>)),
    ('pottedplant',
     0.6630171130015583,
     tensor(58.4880, grad_fn=<SubBackward0>),
     tensor(-59.6517, grad_fn=<SubBackward0>),
     tensor(79.2529, grad_fn=<AddBackward0>),
     tensor(63.0036, grad_fn=<AddBackward0>))],
   'boat': [('boat',
     0.5382174977064302,
     tensor(101.7207, grad_fn=<SubBackward0>),
     tensor(47.0326, grad_fn=<SubBackward0>),
     tensor(200.2770, grad_fn=<AddBackward0>),
     tensor(-38.5687, grad_fn=<AddBackward0>))],
   'bird': [('bird',
     0.5651410619335415,
     tensor(-11.8667, grad_fn=<SubBackward0>),
     tensor(115.7452, grad_fn=<SubBackward0>),
     tensor(87.3825, grad_fn=<AddBackward0>),
     tensor(148.0901, grad_fn=<AddBackward0>))],
   'train': [('train',
     0.6659734384

In [12]:
y_batch.shape, y_batch, y_batch[0].shape

(torch.Size([8, 7, 7, 30]),
 tensor([[[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
 
          [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.000

In [13]:
(y_batch[0].flatten(0, 1))[29]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.])

In [14]:
from src.configs import IDX_TO_CLASS

def find_objects(y_batch, device):
    y_batch = y_batch.flatten(1, 2)
    truth_objects = []

    for target in y_batch:
        class_objects = {}
        
        for cell in target:
            class_name = IDX_TO_CLASS[int(torch.argmax(cell[:C]))]

            if cell[C+4] == 1:
                # the additional 0 is to classify that specific object as 
                # unmatched with a prediction. 1 is for matched. 
                bboxes = torch.cat((cell[C:C+4], torch.tensor([0]).to(device)))
                if class_name in class_objects:
                    class_objects[class_name].append(bboxes)
                else:
                    class_objects[class_name] = [bboxes]

        truth_objects.append(class_objects)
        
    return truth_objects

truth_objects = find_objects(y_batch, "cpu")

In [15]:
len(truth_objects)

8

In [16]:
final_preds[0]

{}

In [18]:
truth_objects

[{'sofa': [tensor([0.3125, 0.1562, 0.5000, 0.2946, 0.0000]),
   tensor([0.1250, 0.3750, 0.4330, 0.4732, 0.0000])]},
 {'dog': [tensor([0.6562, 0.3750, 0.5223, 0.4018, 0.0000])]},
 {'car': [tensor([0.4062, 0.1875, 0.9732, 0.5804, 0.0000])]},
 {'aeroplane': [tensor([0.2812, 0.4062, 0.6429, 0.9643, 0.0000]),
   tensor([0.6875, 0.9375, 0.4018, 0.3705, 0.0000])]},
 {'train': [tensor([0.9062, 0.0625, 0.8214, 0.8393, 0.0000])]},
 {'person': [tensor([0.3438, 0.4688, 0.0982, 0.3839, 0.0000])],
  'motorbike': [tensor([0.3438, 0.0938, 0.1920, 0.3839, 0.0000]),
   tensor([0.5625, 0.0312, 0.2857, 0.5670, 0.0000]),
   tensor([0.4375, 0.8750, 0.5179, 0.8170, 0.0000])]},
 {'bird': [tensor([0.9062, 0.9688, 0.2054, 0.4152, 0.0000])]},
 {'person': [tensor([0.7500, 0.1250, 0.7679, 0.6964, 0.0000])]}]

In [19]:
truth_objects[0]

{'sofa': [tensor([0.3125, 0.1562, 0.5000, 0.2946, 0.0000]),
  tensor([0.1250, 0.3750, 0.4330, 0.4732, 0.0000])]}

In [20]:
TP_IOU_THRESHOLD = 0.5

def tp_fp_and_count_objects(final_preds, truth_objects, all_tp_fp_by_class, class_object_totals):
    # 1. iterate through each image prediction/label in the batch
    for b in range(len(final_preds)):
        img_preds = final_preds[b]
        objects = truth_objects[b]

        # 2. iterate through each class
        for class_name, preds in img_preds.items():            
            # a. if any of the ground truth objects belong to the class
            if class_name in objects:
                # iterate through each prediction, find max IoU truth object, and
                # if the max IoU surpasses the threshold, it is a TP, else FP
                for pred in preds:
                    IoUs = [IoU(object_[0:4], pred[2:6]) for object_ in objects[class_name]]
                    max_idx = IoUs.index(max(IoUs))
                    
                    if IoUs[max_idx] < TP_IOU_THRESHOLD:
                        all_tp_fp_by_class[class_name].append((pred[1], False))
                        
                    elif objects[class_name][max_idx][-1] == 0:
                        all_tp_fp_by_class[class_name].append((pred[1], True))
                        objects[class_name][max_idx][-1] = 1
                        
                    else:
                        all_tp_fp_by_class[class_name].append((pred[1], False))
            else:
                # b. all predictions belonging to class are FP since there are
                # no ground truth objects belonging to that class
                for pred in preds:
                    all_tp_fp_by_class[class_name].append((pred[1], False))    

        # 3. add the number of objects that belong to each class to the total
        for class_name, object_list in objects.items():
            class_object_totals[class_name] += len(object_list)

In [21]:
all_tp_fp_by_class = {
    "aeroplane": [],
    "bicycle": [],
    "bird": [],
    "boat": [],
    "bottle": [],
    "bus": [],
    "car": [],
    "cat": [],
    "chair": [],
    "cow":[],
    "diningtable": [],
    "dog": [],
    "horse": [],
    "motorbike": [],
    "person": [],
    "pottedplant": [],
    "sheep": [],
    "sofa": [],
    "train": [],
    "tvmonitor": [],
}

class_object_totals = {
    "aeroplane": 0,
    "bicycle": 0,
    "bird": 0,
    "boat": 0,
    "bottle": 0,
    "bus": 0,
    "car": 0,
    "cat": 0,
    "chair": 0,
    "cow":0,
    "diningtable": 0,
    "dog": 0,
    "horse": 0,
    "motorbike": 0,
    "person": 0,
    "pottedplant": 0,
    "sheep": 0,
    "sofa": 0,
    "train": 0,
    "tvmonitor": 0,
}

In [22]:
tp_fp_and_count_objects(final_preds, truth_objects, all_tp_fp_by_class, class_object_totals)

all_tp_fp_by_class["bird"].sort(key=itemgetter(0), reverse=True)
all_tp_fp_by_class, class_object_totals

({'aeroplane': [],
  'bicycle': [],
  'bird': [(0.5651410619335415, False)],
  'boat': [(0.5382174977064302, False), (0.6664213618214809, False)],
  'bottle': [(0.42538744741804635, False)],
  'bus': [],
  'car': [],
  'cat': [],
  'chair': [],
  'cow': [],
  'diningtable': [(0.5390705336079336, False)],
  'dog': [(0.3791735284010116, False)],
  'horse': [],
  'motorbike': [],
  'person': [],
  'pottedplant': [(0.4092096754303718, False),
   (0.6630171130015583, False),
   (0.4416272593814625, False)],
  'sheep': [],
  'sofa': [],
  'train': [(0.6659734384729461, False), (0.40654517155977743, False)],
  'tvmonitor': [(0.8925000087421573, False)]},
 {'aeroplane': 2,
  'bicycle': 0,
  'bird': 1,
  'boat': 0,
  'bottle': 0,
  'bus': 0,
  'car': 1,
  'cat': 0,
  'chair': 0,
  'cow': 0,
  'diningtable': 0,
  'dog': 1,
  'horse': 0,
  'motorbike': 3,
  'person': 2,
  'pottedplant': 0,
  'sheep': 0,
  'sofa': 2,
  'train': 1,
  'tvmonitor': 0})